# Saliendo de lo Pandito v4
## Módulo 7: Series de Tiempo Económicas y Financieras

### Objetivos del Módulo:
1. Manipular objetos `DatetimeIndex` y remuestreo de frecuencias (`resample`).
2. Deflactar series financieras históricas ajustándolas por la tasa de inflación (IPC).
3. Calcular promedios móviles y la Tasa de Crecimiento Anual Compuesto (**CAGR**).


In [0]:
# 🎯 OPCIONAL: Usar datos reales de Unity Catalog
# Si ejecutaste el notebook 00_05_Preparacion_Datos_Empresariales.ipynb,
# puedes cargar los datos reales de Los Andes Market.

# Descomentar para usar datos reales:
"""
print("💾 Cargando datos reales desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    
    # Preparar para series de tiempo
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    df_ventas = df_ventas.sort_values('fecha')
    
    # Opción 1: Usar agregado mensual de todas las sucursales
    df_ts_real = df_ventas.groupby('fecha')['ventas'].sum().reset_index()
    df_ts_real = df_ts_real.set_index('fecha')
    df_ts_real.columns = ['Ventas_Totales']
    
    # Opción 2: Usar serie de una sucursal específica
    suc_id = 'SUC001'  # Centro - San Martín
    df_ts_sucursal = df_ventas[df_ventas['sucursal_id'] == suc_id].set_index('fecha')[['ventas']]
    df_ts_sucursal.columns = ['Ventas_SUC001']
    
    print(f"✅ Datos reales cargados:")
    print(f"   • Período: {df_ventas['fecha'].min().date()} a {df_ventas['fecha'].max().date()}")
    print(f"   • Total meses: {len(df_ts_real)}")
    print(f"   • Sucursales: {df_ventas['sucursal_id'].nunique()}")
    print(f"\n💡 Variables disponibles:")
    print(f"   • df_ts_real: Serie agregada de todas las sucursales")
    print(f"   • df_ts_sucursal: Serie de {suc_id}")
    print(f"   • df_ventas: DataFrame completo con todas las sucursales")
    
    # Visualizar muestra
    print(f"\n📊 Muestra de serie temporal agregada:")
    display(df_ts_real.head(10))
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print("   Continuando con datos sintéticos del notebook...")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("   Para usar datos reales, descomenta el bloque de arriba")
print("="*70)

In [0]:
import pandas as pd
import numpy as np

fechas = pd.date_range(start='2023-01-01', periods=12, freq='ME')
ventas_nominales = [100, 110, 125, 130, 145, 160, 175, 190, 210, 230, 250, 280]
ipc_acumulado = [1.00, 1.06, 1.13, 1.20, 1.28, 1.37, 1.46, 1.56, 1.68, 1.80, 1.93, 2.07]

df_ts = pd.DataFrame({'Ventas_Nominales': ventas_nominales, 'IPC': ipc_acumulado}, index=fechas)

# Deflactación: Ventas Reales a moneda constante
df_ts['Ventas_Reales'] = df_ts['Ventas_Nominales'] / df_ts['IPC']
df_ts['Promedio_Movil_3M'] = df_ts['Ventas_Reales'].rolling(window=3).mean()

print("--- Serie de Ventas Ajustada por Inflación ---")
print(df_ts[['Ventas_Nominales', 'Ventas_Reales', 'Promedio_Movil_3M']])

